In [0]:
%run  /Workspace/Users/rana@ghazzi.com/Dev/wikimedia/resources/notebooks/params_config

In [0]:

window= Window.partitionBy("wiki", "title_url").orderBy(col("timestamp").desc())

df = (spark.read.table(f"{catalog}.{bronze_schema}.{bronze_table}")
          .filter((col("bot") == False) & col("type").isin("edit", "new"))
          .select("id", "title", "comment", "title_url", "timestamp", "type", "user", "wiki")
          .withColumn("rn", row_number().over(window))
          .filter("rn = 1")
          .drop("rn")
          .createOrReplaceTempView("wikimedia"))

In [0]:
%sql
SElect* from wikimedia
where  type = 'new'
--and bot = false

In [0]:
%sql

select count(id) over( partition by title_url order by timestamp desc) as cnt, (*)
from workspace.bronze.wikimedia_bronze
where bot = false
and type in ('edit','new')
order by title_url,timestamp,id

In [0]:
from pyspark.sql.functions import col, count, row_number, current_timestamp
from pyspark.sql.window import Window
duplicate_titles = (spark.read
             .table(f"{catalog}.{bronze_schema}.{bronze_table}")
             .filter((col("bot") == False) & (col("type").isin("edit", "new")))
             .select("id", "title", "comment", "title_url", "timestamp", "type", "user", "wiki")
             .groupBy("title","wiki","title_url","timestamp")
             .agg(count("id").alias("count"))
             .filter(col("count") > 1)).display()